## 自然言語処理モデルを構築するための準備段階

In [ ]:
!pip install keras-core --upgrade
!pip install -q keras-nlp --upgrade

# This sample uses Keras Core, the multi-backend version of Keras.
# The selected backend is TensorFlow (other supported backends are 'jax' and 'torch')
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

## 必要なライブラリのインポート、環境のバージョン確認

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import keras_core as keras
import keras_nlp
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

## データ読み込み、訓練データとテストデータのサイズ確認

In [ ]:
train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

print('Training Set Shape = {}'.format(train.shape))
print('Training Set Memory Usage = {:.2f} MB'.format(train.memory_usage().sum() / 1024**2))
print('Test Set Shape = {}'.format(test.shape))
print('Test Set Memory Usage = {:.2f} MB'.format(test.memory_usage().sum() / 1024**2))

In [ ]:
train.head()

## 各ツイート本文（text）の文字数（長さ）を調べて統計的に把握

In [ ]:
train["length"] = train["text"].apply(lambda x : len(x))
test["length"] = test["text"].apply(lambda x : len(x))

print("Train Length Stat")
print(train["length"].describe())
print()

print("Test Length Stat")
print(test["length"].describe())

## 前処理

In [ ]:
def preprocess(text):
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  # URL除去
    text = re.sub(r"<.*?>", "", text)  # HTML除去
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

In [ ]:
train["text"] = train["text"].apply(preprocess)

In [ ]:
X = train["text"].values  
y = train["target"].values

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## モデルの学習準備

In [ ]:
from keras_nlp.models import RobertaClassifier
from keras_nlp.models import RobertaPreprocessor
from keras_core.optimizers import Adam

In [ ]:
preset = "roberta_base_en"
preprocessor = keras_nlp.models.RobertaPreprocessor.from_preset(
    preset,
    sequence_length=160
)
classifier = keras_nlp.models.RobertaClassifier.from_preset(
    preset,
    preprocessor=preprocessor,
    num_classes=2
)

In [ ]:
classifier.trainable = True

## サンプルデータ学習

In [ ]:
classifier.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    metrics=["accuracy"]
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

In [ ]:
classifier.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=4,
    class_weight=class_weights
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def displayConfusionMatrix(y_true, y_pred_proba, dataset="Dataset"):
    # softmax出力（確率）からラベルに変換
    y_pred_labels = np.argmax(y_pred_proba, axis=1)

    # 混同行列
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred_labels,
        display_labels=["Not Disaster", "Disaster"],
        cmap=plt.cm.Blues
    )
    f1 = f1_score(y_true, y_pred_labels)
    disp.ax_.set_title(f"Confusion Matrix on {dataset} -- F1 Score: {f1:.4f}")
    plt.show()

In [ ]:
y_pred_val_proba = classifier.predict(X_val)
y_pred_labels = np.argmax(y_pred_val_proba, axis=1)

In [ ]:
unique, counts = np.unique(y_pred_labels, return_counts=True)
print("予測ラベルの分布:")
for label, count in zip(unique, counts):
    print(f"Label {label}: {count} 件")

In [ ]:
# 検証データの予測
y_pred_val_proba = classifier.predict(X_val)

displayConfusionMatrix(y_val, y_pred_val_proba, dataset="Validation")

## テストデータ

In [ ]:
train["text"] = train["text"].apply(preprocess)
test["text"] = test["text"].apply(preprocess)

X_test = test["text"]

In [ ]:
y_pred_test = classifier.predict(X_test)
y_pred_labels = np.argmax(y_pred_test, axis=1)

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "target": y_pred_labels
})

In [ ]:
submission.to_csv("submission.csv", index=False)

submission.head()